# ORPO training release copy

This copy was prepared from the author-supplied training notebook. Stored outputs and widget metadata were removed. A private dataset identifier was replaced with the `CAEE_DATASET_REPO` environment variable. Credentials must be provided through the execution environment; no credentials are included.

External experiment logging and automatic Hub publication are disabled. Merged model artifacts are saved locally. Model settings, selected dataset size, chat template, and the original 1% internal validation split are retained.

This notebook has been checked structurally and for Python syntax but has not been executed during release preparation. It requires a CUDA GPU, access to the specified base model, and an appropriately structured dataset to which the user has authorized access. Read the accompanying README before running.


# JP1_ORPO Training

This notebook is designed to train a language model on the ORPO dataset. The tasks performed in this notebook include:

1. **Setup**: Installing and importing necessary libraries for model training.
2. **Data Loading and Preparation**: Loading the augmented ORPO dataset and splitting it into training and validation sets.
3. **Tokenization**: Tokenizing the dataset using a pre-trained tokenizer to prepare it for model training.
4. **Model Initialization**: Loading a pre-trained language model to fine-tune on the ORPO dataset.
5. **Training Configuration**: Defining training arguments and setting up the training environment.
6. **Data Collation**: Creating a data collator to handle language modeling tasks during training.
7. **Model Training**: Training the language model on the training dataset.
8. **Model Saving**: Saving the fine-tuned model and tokenizer for future use in downstream tasks.

In [ ]:
!pip install -qqq -U pandas==2.1.4 transformers datasets accelerate peft trl bitsandbytes wandb tqdm huggingface_hub requests==2.31.0 fsspec==2024.3.1 gcsfs==2024.3.1 pyarrow==16.0.0

In [ ]:
import gc
import os
import re
import torch
import wandb
import transformers
import datasets
import shutil
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset, Value, Features, Sequence
from datasets import load_dataset
from huggingface_hub import login, HfApi
from sklearn.model_selection import train_test_split
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import ORPOConfig, ORPOTrainer, setup_chat_format
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)


# Fail before loading model weights if the private dataset input is absent.
if not os.environ.get("CAEE_DATASET_REPO"):
    raise ValueError("Set CAEE_DATASET_REPO before loading model weights.")

# Authentication is supplied by the execution environment.
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)


In [ ]:
# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

In [ ]:
# Model
base_model = "meta-llama/Meta-Llama-3-8B-Instruct"
# base_model = "meta-llama/Meta-Llama-3-8B"
new_model = "ie_Llama-3-8B"

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)
model, tokenizer = setup_chat_format(model, tokenizer)
model = prepare_model_for_kbit_training(model)

In [ ]:
dataset_name = os.environ.get("CAEE_DATASET_REPO")
if not dataset_name:
    raise ValueError("Set CAEE_DATASET_REPO to a dataset repository you are authorized to access.")

dataset = load_dataset(dataset_name, split="train")
dataset = dataset.shuffle(seed=42).select(range(4096))

tokenizer.chat_template = "{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"

def format_chat_template(row):
    row["chosen"] = tokenizer.apply_chat_template(row["chosen"], tokenize=False)
    row["rejected"] = tokenizer.apply_chat_template(row["rejected"], tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    num_proc= os.cpu_count(),
)
dataset = dataset.train_test_split(test_size=0.01)

In [ ]:
# Report sizes without printing participant-level records.
print({"training_rows": len(dataset["train"]), "validation_rows": len(dataset["test"])})


In [ ]:
orpo_args = ORPOConfig(
    learning_rate=8e-6,
    beta=0.1,
    lr_scheduler_type="linear",
    max_length=1024,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    num_train_epochs=1,
    evaluation_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    report_to="none",
    output_dir="./results/",
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    tokenizer=tokenizer,
)
trainer.train()
trainer.save_model(new_model)

# Flush memory
del trainer, model
gc.collect()
torch.cuda.empty_cache()

# Reload tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
)

# Manually move model to GPU
model = model.to('cuda')

# Setup chat format (if necessary)
model, tokenizer = setup_chat_format(model, tokenizer)

# Merge adapter with base model
adapter = PeftModel.from_pretrained(model, new_model)
adapter = adapter.to('cuda')  # Ensure adapter is on the same device

# Merge and unload
model = adapter.merge_and_unload()

# Save locally; model publication is outside this release-copy workflow.
merged_output_dir = new_model + "_merged"
model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)